# Phase 9 — Qualitative Analysis

Collects samples where (A) Baseline was wrong and (B) Ours was correct,
then prints clip path + gaze_vec + prediction comparison for manual inspection.

Requires:
  - `results/eval_all.json`
  - `features/gaze/test.pkl`

In [ ]:
import json
import pickle
import sys
import csv
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('..')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'TelME' / 'MELD'))

EMOTION_LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
FEAT_NAMES = ['p_face', 'p_out', 'switch_rate', 'entropy', 'target_count', 'p_center']

with open(ROOT / 'results' / 'eval_all.json') as f:
    results = json.load(f)
print('Available conditions:', list(results.keys()))

In [ ]:
# Re-run inference to get per-sample predictions
# (requires checkpoints and GPU/MPS)
#
# If you already have pred arrays saved, load them here instead.

print("TODO: load per-sample predictions from eval or re-run inference.")
print("Placeholder: using dummy data for structure demonstration.")

N = 100
np.random.seed(42)
y_true  = np.random.randint(0, 7, N)
pred_A  = np.random.randint(0, 7, N)
pred_B  = y_true.copy()
pred_B[:80] = np.random.randint(0, 7, 80)   # B gets ~20% right that A got wrong

In [ ]:
# Collect A-wrong, B-correct samples
a_wrong_b_right = np.where((pred_A != y_true) & (pred_B == y_true))[0]
print(f'Samples where A is wrong and B is correct: {len(a_wrong_b_right)}')
sample_idx = a_wrong_b_right[:20]   # up to 20 cases

In [ ]:
# Load test CSV for clip paths
csv_path = ROOT / 'data' / 'MELD.Raw' / 'test_sent_emo.csv'
if not csv_path.exists():
    csv_path = ROOT / 'data' / 'MELD.Raw' / 'test_meld_emo.csv'

rows_csv = []
with csv_path.open() as f:
    for row in csv.DictReader(f):
        rows_csv.append(row)
print(f'Test CSV rows: {len(rows_csv)}')

In [ ]:
# Load gaze features
gaze_path = ROOT / 'features' / 'gaze' / 'test.pkl'
if gaze_path.exists():
    with gaze_path.open('rb') as f:
        gaze_feats = pickle.load(f)
else:
    gaze_feats = {}
    print('[WARNING] Test gaze features not found.')

In [ ]:
# Build comparison table
records = []
for idx in sample_idx:
    if idx >= len(rows_csv):
        continue
    row = rows_csv[idx]
    d, u = int(row.get('Dialogue_ID', -1)), int(row.get('Utterance_ID', -1))
    gvec = gaze_feats.get((d, u), np.zeros(6))
    records.append({
        'sample_idx':    int(idx),
        'dialogue':      d,
        'utterance':     u,
        'true_emotion':  EMOTION_LABELS[y_true[idx]],
        'pred_A':        EMOTION_LABELS[pred_A[idx]],
        'pred_B':        EMOTION_LABELS[pred_B[idx]],
        'clip_path':     row.get('Video_Path', ''),
        **{FEAT_NAMES[i]: round(float(gvec[i]),3) for i in range(6)},
    })

df_qual = pd.DataFrame(records)
display(df_qual)

In [ ]:
# Save to error_analysis.md
out_path = ROOT / 'results' / 'error_analysis.md'
with out_path.open('w') as f:
    f.write('# Error Analysis: A-wrong, B-correct samples\n\n')
    f.write(df_qual.to_markdown(index=False))
    f.write('\n\n## Notes\n')
    f.write('- Gaze features are z-score normalised (train statistics).\n')
    f.write('- p_face high + pred_B correct → gaze-face correlation helps.\n')
print(f'Saved: {out_path}')